# Erythroid differential-expression analysis

Run this notebook from the repository root and execute cells from top to bottom. All inputs use repository-relative paths. Generated files are written below `data/processed/` or `results/`. Outputs and execution counters are cleared in the version-controlled copy.


In [ ]:
# Step 0: resolve repository-relative paths.
project_root <- normalizePath(Sys.getenv("BMO_PROJECT_ROOT", unset = "."), winslash = "/", mustWork = TRUE)
dir.create(file.path(project_root, "data", "processed"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "tables"), recursive = TRUE, showWarnings = FALSE)

library(Seurat)
library(SeuratObject)
library(tidyverse)
library(Matrix)
library(pheatmap)


<b><font size=5 color=pink >Step 1: prepare haematopoietic data</font></b>


In [ ]:
Haematopoietic_merged <- readRDS(file.path(project_root, "data", "processed", "BMOs_Haematopoietic_Final_Integrated_CCA.rds"))


In [ ]:
table(Haematopoietic_merged@meta.data$celltype, useNA = "ifany")


In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)

# ============================================================
# ============================================================

Haematopoietic_merged$celltype <- as.character(Haematopoietic_merged$celltype)
Haematopoietic_merged$dataset <- as.character(Haematopoietic_merged$dataset)

Haematopoietic_merged$group_plot <- as.character(Haematopoietic_merged$group)

idx_na <- is.na(Haematopoietic_merged$group_plot) |
  Haematopoietic_merged$group_plot == ""

Haematopoietic_merged$group_plot[idx_na] <-
  Haematopoietic_merged$dataset[idx_na]

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("Adult_BM", "AdultBM", "Adult BM")
] <- "ABM"

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("FBM", "Fetal_BM", "Fetal BM")
] <- "FBM"

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("Organoids23", "BMO2023", "BMO-2023")
] <- "mBMO"

table(Haematopoietic_merged$group_plot, useNA = "ifany")


In [ ]:
# ============================================================
# ============================================================

Haematopoietic_merged$celltype_broad <- dplyr::case_when(

  # HSPC / progenitor
  Haematopoietic_merged$celltype %in% c(
    "HSC",
    "MPP",
    "HSPC",
    "Cycling HSPC",
    "HSC/MPP and pro",
    "CLP",
    "MEP",
    "GMP",
    "Early Myeloid Progenitor",
    "Myeloid Progenitor"
  ) ~ "HSPC/Progenitor",

  # Erythroid
  Haematopoietic_merged$celltype %in% c(
    "Erythroid",
    "erythroid",
    "Erythroblast",
    "Late Erythroid",
    "RBC"
  ) ~ "Erythroid",

  # Megakaryocyte
  Haematopoietic_merged$celltype %in% c(
    "Megakaryocyte",
    "MK"
  ) ~ "Megakaryocyte",

  # Monocyte / macrophage
  Haematopoietic_merged$celltype %in% c(
    "Monocyte",
    "monocyte",
    "Macrophage",
    "Macrophages"
  ) ~ "Monocyte/Macrophage",

  # Granulocyte / mast / basophil / eosinophil
  Haematopoietic_merged$celltype %in% c(
    "Neutrophil",
    "neutrophil",
    "Basophil",
    "Eosinophil",
    "Mast",
    "Ba/Eo/Ma",
    "eo/baso/mast",
    "Late Myeloid"
  ) ~ "Granulocyte/Mast",

  # DC
  Haematopoietic_merged$celltype %in% c(
    "DC",
    "pDC",
    "Cycling DCs"
  ) ~ "DC",

  # B lineage
  Haematopoietic_merged$celltype %in% c(
    "B_lineage",
    "Pre-Pro B",
    "Pro-B",
    "Pre-B",
    "Mature B"
  ) ~ "B lineage",

  # Plasma cell
  Haematopoietic_merged$celltype %in% c(
    "Plasma Cell"
  ) ~ "Plasma cell",

  # T / NK
  Haematopoietic_merged$celltype %in% c(
    "CD4+ T-Cell",
    "CD8+ T-Cell",
    "T_NK"
  ) ~ "T/NK",

  TRUE ~ "Other"
)

table(Haematopoietic_merged$celltype, Haematopoietic_merged$celltype_broad, useNA = "ifany")
table(Haematopoietic_merged$celltype_broad, useNA = "ifany")


<b><font size=5 color=pink >Step 2: differential-expression analysis</font></b>


In [ ]:
# ============================================================
# Haematopoietic major celltype DEG analysis
# ABM vs Dynamic.25d
# ============================================================

library(Seurat)
library(dplyr)
library(tidyr)
library(ggplot2)

# ============================================================
# ============================================================

DefaultAssay(Haematopoietic_merged) <- "RNA"

group_col <- "group_plot"
celltype_col <- "celltype_broad"

min_cells <- 30
logfc_cutoff <- 1

Haematopoietic_merged@meta.data[[celltype_col]] <- as.character(
  Haematopoietic_merged@meta.data[[celltype_col]]
)

Haematopoietic_merged@meta.data[[group_col]] <- as.character(
  Haematopoietic_merged@meta.data[[group_col]]
)

table(Haematopoietic_merged@meta.data[[group_col]], useNA = "ifany")
table(Haematopoietic_merged@meta.data[[celltype_col]], useNA = "ifany")


# ============================================================
# ============================================================

meta_deg <- Haematopoietic_merged@meta.data %>%
  dplyr::filter(
    !is.na(.data[[celltype_col]]),
    .data[[celltype_col]] != "Other",
    !is.na(.data[[group_col]])
  )

diag_tbl <- meta_deg %>%
  dplyr::count(
    celltype_major = .data[[celltype_col]],
    group_plot_use = .data[[group_col]],
    name = "n"
  ) %>%
  tidyr::pivot_wider(
    names_from = group_plot_use,
    values_from = n,
    values_fill = 0
  )

for (g in c("ABM", "Dynamic.25d")) {
  if (!g %in% colnames(diag_tbl)) {
    diag_tbl[[g]] <- 0
  }
}

diag_tbl <- diag_tbl %>%
  dplyr::mutate(
    ABM_OK = ABM >= min_cells,
    Dynamic25d_OK = `Dynamic.25d` >= min_cells,
    will_run_DEG = ABM_OK & Dynamic25d_OK
  ) %>%
  dplyr::arrange(desc(will_run_DEG), celltype_major)

print(diag_tbl, n = 50)

valid_types <- diag_tbl %>%
  dplyr::filter(will_run_DEG) %>%
  dplyr::pull(celltype_major)

skip_types <- diag_tbl %>%
  dplyr::filter(!will_run_DEG) %>%
  dplyr::pull(celltype_major)

message(
  " Cell types retained for differential-expression analysis: ",
  paste(valid_types, collapse = ", ")
)

if (length(skip_types) > 0) {
  message(
    "Warning: The following cell types were skipped because one group contained fewer than ",
    min_cells,
    " cells: ",
    paste(skip_types, collapse = ", ")
  )
}


# ============================================================
# ============================================================

major_DEG_list <- list()

for (ct in valid_types) {

  message("Running DEG for: ", ct)

  cells_use <- rownames(Haematopoietic_merged@meta.data)[
    Haematopoietic_merged@meta.data[[celltype_col]] == ct &
      Haematopoietic_merged@meta.data[[group_col]] %in% c("ABM", "Dynamic.25d")
  ]

  obj_ct <- subset(
    Haematopoietic_merged,
    cells = cells_use
  )

  DefaultAssay(obj_ct) <- "RNA"
  Idents(obj_ct) <- group_col

  ct_count <- table(Idents(obj_ct))

  if (
    all(c("ABM", "Dynamic.25d") %in% names(ct_count)) &&
    ct_count["ABM"] >= min_cells &&
    ct_count["Dynamic.25d"] >= min_cells
  ) {

    deg <- FindMarkers(
      obj_ct,
      ident.1 = "ABM",
      ident.2 = "Dynamic.25d",
      assay = "RNA",
      slot = "data",
      only.pos = FALSE,
      min.pct = 0.25,
      logfc.threshold = 0
    )

    if (!"avg_log2FC" %in% colnames(deg) && "avg_logFC" %in% colnames(deg)) {
      deg$avg_log2FC <- deg$avg_logFC
    }

    deg$gene <- rownames(deg)
    deg$celltype_major <- ct
    deg$n_ABM <- as.numeric(ct_count["ABM"])
    deg$n_Dynamic25d <- as.numeric(ct_count["Dynamic.25d"])

    major_DEG_list[[ct]] <- deg

  } else {

    message(
      "Skipped ", ct,
      " because ABM or Dynamic.25d cells < ",
      min_cells
    )
  }
}


# ============================================================
# ============================================================

if (length(major_DEG_list) == 0) {
  stop("No cell type contains at least min_cells cells in both comparison groups.")
}

Haematopoietic_major_ABM_vs_Dynamic25d_DEG <- dplyr::bind_rows(
  major_DEG_list,
  .id = "celltype_from_list"
)

head(Haematopoietic_major_ABM_vs_Dynamic25d_DEG)


In [ ]:
# ============================================================
#    pct_diff = pct.1 - pct.2 = ABM - Dynamic.25d
# ============================================================

logfc_cutoff <- 2
pct_cutoff <- 0.2

Haematopoietic_major_ABM_vs_Dynamic25d_DEG <-
  Haematopoietic_major_ABM_vs_Dynamic25d_DEG %>%
  dplyr::mutate(
    pct_diff = pct.1 - pct.2,
    DEG_group = dplyr::case_when(
      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC >= logfc_cutoff &
        pct_diff >= pct_cutoff ~ "Higher in ABM",

      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC <= -logfc_cutoff &
        pct_diff <= -pct_cutoff ~ "Higher in Dynamic.25d",

      TRUE ~ "Not significant"
    )
  )

table(Haematopoietic_major_ABM_vs_Dynamic25d_DEG$DEG_group)


# ============================================================
# ============================================================

saveRDS(
  Haematopoietic_major_ABM_vs_Dynamic25d_DEG,
  file = file.path(project_root, "results", "tables", "Haematopoietic_major_ABM_vs_Dynamic25d_DEG_logFC1_pct005.rds")
)

write.csv(
  Haematopoietic_major_ABM_vs_Dynamic25d_DEG,
  file = file.path(project_root, "results", "tables", "Haematopoietic_major_ABM_vs_Dynamic25d_DEG_logFC1_pct005.csv"),
  row.names = FALSE
)


In [ ]:
# ============================================================
# ============================================================

logfc_cutoff <- 1
pct_cutoff <- 0.2

deg_df <- Haematopoietic_major_ABM_vs_Dynamic25d_DEG %>%
  dplyr::mutate(
    pct_diff = pct.1 - pct.2,
    DEG_group = dplyr::case_when(
      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC >= logfc_cutoff &
        pct_diff >= pct_cutoff ~ "Higher in ABM",

      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC <= -logfc_cutoff &
        pct_diff <= -pct_cutoff ~ "Higher in Dynamic.25d",

      TRUE ~ "Not significant"
    ),
    DEG_group = factor(
      DEG_group,
      levels = c(
        "Not significant",
        "Higher in ABM",
        "Higher in Dynamic.25d"
      )
    )
  )

table(deg_df$DEG_group)

deg_df %>%
  dplyr::filter(
    DEG_group == "Higher in ABM" &
      !(avg_log2FC >= logfc_cutoff & pct_diff >= pct_cutoff)
  ) %>%
  nrow()

deg_df %>%
  dplyr::filter(
    DEG_group == "Higher in Dynamic.25d" &
      !(avg_log2FC <= -logfc_cutoff & pct_diff <= -pct_cutoff)
  ) %>%
  nrow()

deg_count_tbl <- deg_df %>%
  dplyr::count(celltype_major, DEG_group, name = "n_genes") %>%
  tidyr::pivot_wider(
    names_from = DEG_group,
    values_from = n_genes,
    values_fill = 0
  )

if (!"Higher in ABM" %in% colnames(deg_count_tbl)) {
  deg_count_tbl$`Higher in ABM` <- 0
}

if (!"Higher in Dynamic.25d" %in% colnames(deg_count_tbl)) {
  deg_count_tbl$`Higher in Dynamic.25d` <- 0
}

if (!"Not significant" %in% colnames(deg_count_tbl)) {
  deg_count_tbl$`Not significant` <- 0
}

deg_count_tbl <- deg_count_tbl %>%
  dplyr::mutate(
    Total_DEG = `Higher in ABM` + `Higher in Dynamic.25d`,
    label_ABM = paste0("ABM: ", `Higher in ABM`),
    label_Dynamic = paste0("Dynamic: ", `Higher in Dynamic.25d`)
  ) %>%
  dplyr::arrange(desc(Total_DEG))

deg_count_tbl


In [ ]:
# ============================================================
# ============================================================

library(dplyr)
library(tidyr)
library(ggplot2)

# ============================================================
# ============================================================

logfc_cutoff <- 1
pct_cutoff <- 0.2

# ============================================================
# ============================================================

deg_df <- Haematopoietic_major_ABM_vs_Dynamic25d_DEG %>%
  dplyr::mutate(
    pct_diff = pct.1 - pct.2,

    DEG_group = dplyr::case_when(
      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC >= logfc_cutoff &
        pct_diff >= pct_cutoff ~ "Higher in ABM",

      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC <= -logfc_cutoff &
        pct_diff <= -pct_cutoff ~ "Higher in Dynamic.25d",

      TRUE ~ "Not significant"
    ),

    DEG_group = factor(
      DEG_group,
      levels = c(
        "Not significant",
        "Higher in ABM",
        "Higher in Dynamic.25d"
      )
    )
  )

table(deg_df$DEG_group)

# ============================================================
# ============================================================

celltype_order <- c(
  "HSPC/Progenitor",
  "Erythroid",
  "Megakaryocyte",
  "Granulocyte/Mast",
  "Monocyte/Macrophage",
  "DC",
  "B lineage",
  "Plasma cell",
  "T/NK"
)

celltype_order_use <- celltype_order[
  celltype_order %in% unique(deg_df$celltype_major)
]

deg_df$celltype_major <- factor(
  deg_df$celltype_major,
  levels = celltype_order_use
)

# ============================================================
# ============================================================

deg_count_tbl <- deg_df %>%
  dplyr::count(celltype_major, DEG_group, name = "n_genes") %>%
  tidyr::pivot_wider(
    names_from = DEG_group,
    values_from = n_genes,
    values_fill = 0
  )

if (!"Higher in ABM" %in% colnames(deg_count_tbl)) {
  deg_count_tbl$`Higher in ABM` <- 0
}

if (!"Higher in Dynamic.25d" %in% colnames(deg_count_tbl)) {
  deg_count_tbl$`Higher in Dynamic.25d` <- 0
}

if (!"Not significant" %in% colnames(deg_count_tbl)) {
  deg_count_tbl$`Not significant` <- 0
}

deg_count_tbl <- deg_count_tbl %>%
  dplyr::mutate(
    Total_DEG = `Higher in ABM` + `Higher in Dynamic.25d`,
    label_ABM = paste0("ABM: ", `Higher in ABM`),
    label_Dynamic = paste0("Dynamic: ", `Higher in Dynamic.25d`)
  ) %>%
  dplyr::arrange(desc(Total_DEG))

deg_count_tbl

# ============================================================
# ============================================================

deg_cols <- c(
  "Higher in ABM"         = "#D95F5F",
  "Higher in Dynamic.25d" = "#6A9BCB"
)

# ============================================================
# ============================================================

p <- ggplot() +

  geom_hline(
    yintercept = c(-logfc_cutoff, logfc_cutoff),
    linetype = "dashed",
    linewidth = 0.5,
    color = "grey60"
  ) +

  geom_vline(
    xintercept = 0,
    linetype = "solid",
    linewidth = 0.45,
    color = "grey55"
  ) +

  geom_vline(
    xintercept = c(-pct_cutoff, pct_cutoff),
    linetype = "dashed",
    linewidth = 0.45,
    color = "grey70"
  ) +

  geom_point(
    data = deg_df %>% dplyr::filter(DEG_group == "Not significant"),
    aes(x = pct_diff, y = avg_log2FC),
    color = "grey82",
    size = 0.85,
    alpha = 0.55,
    show.legend = FALSE
  ) +

  geom_point(
    data = deg_df %>% dplyr::filter(DEG_group == "Higher in Dynamic.25d"),
    aes(x = pct_diff, y = avg_log2FC, color = DEG_group),
    size = 1.25,
    alpha = 0.9
  ) +

  geom_point(
    data = deg_df %>% dplyr::filter(DEG_group == "Higher in ABM"),
    aes(x = pct_diff, y = avg_log2FC, color = DEG_group),
    size = 1.25,
    alpha = 0.9
  ) +

  geom_text(
    data = deg_count_tbl,
    aes(
      x = 0.95,
      y = Inf,
      label = label_ABM
    ),
    inherit.aes = FALSE,
    hjust = 1,
    vjust = 1.35,
    size = 3.4,
    fontface = "bold",
    color = "#D95F5F"
  ) +

  geom_text(
    data = deg_count_tbl,
    aes(
      x = -0.95,
      y = -Inf,
      label = label_Dynamic
    ),
    inherit.aes = FALSE,
    hjust = 0,
    vjust = -0.8,
    size = 3.4,
    fontface = "bold",
    color = "#6A9BCB"
  ) +

  scale_color_manual(values = deg_cols) +

  coord_cartesian(
    xlim = c(-1, 1)
  ) +

  facet_wrap(
    ~ celltype_major,
    ncol = 5,
    scales = "fixed"
  ) +

  labs(
  x = expression(Delta~"Percentage Difference"),
  y = "Average log2 Fold Change",
  color = NULL,
  caption = paste0(
    "Colored points indicate genes with FDR-adjusted P value < 0.05, ",
    "|log2FC| >= ", logfc_cutoff,
    ", and |delta  percentage difference| >= ", pct_cutoff, "."
  )
)+

  theme_bw(base_size = 14) +
  theme(
    panel.grid = element_blank(),

    strip.background = element_rect(
      fill = "grey92",
      color = NA
    ),
    strip.text = element_text(
      face = "bold",
      size = 12,
      color = "black"
    ),

    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      color = "black"
    ),
    axis.text.y = element_text(
      color = "black"
    ),
    axis.title = element_text(
      color = "black"
    ),

    legend.position = "top",
    legend.text = element_text(
      size = 10,
      color = "black"
    ),

    panel.border = element_rect(
      color = "black",
      linewidth = 0.6
    ),

    plot.caption = element_text(
      size = 9,
      color = "grey30",
      hjust = 0
    ),

    plot.margin = margin(
      t = 5.5,
      r = 8,
      b = 18,
      l = 5.5
    )
  )

p


In [ ]:
# ============================================================
# ============================================================

check_grey_ABM <- deg_df %>%
  dplyr::filter(
    avg_log2FC >= logfc_cutoff,
    pct_diff >= pct_cutoff,
    DEG_group != "Higher in ABM"
  ) %>%
  dplyr::mutate(
    reason = dplyr::case_when(
      is.na(p_val_adj) ~ "p_val_adj is NA",
      p_val_adj >= 0.05 ~ "p_val_adj >= 0.05",
      TRUE ~ "other / DEG_group not updated"
    )
  )

check_grey_ABM %>%
  dplyr::count(celltype_major, reason)

check_grey_ABM %>%
  dplyr::select(
    celltype_major,
    gene,
    avg_log2FC,
    pct.1,
    pct.2,
    pct_diff,
    p_val_adj,
    DEG_group,
    reason
  ) %>%
  dplyr::arrange(celltype_major, desc(avg_log2FC)) %>%
  head(50)


In [ ]:
# ============================================================
# ============================================================

ggsave(
  filename = file.path(project_root, "results", "figures", "Haematopoietic_ABM_vs_Dynamic25d_pctdiff_log2fc_facet_logFC2_pct02.pdf"),
  plot = p,
  width = 8,
  height = 4.2,
  dpi = 300,
  device = cairo_pdf
)


<b><font size=5 color=pink >Step 3: gene-ontology enrichment analysis</font></b>


In [ ]:
# ============================================================
# ============================================================

library(dplyr)

out_dir <- file.path(project_root, "results", "figures", "ABM_up_DEG_by_celltype")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# ============================================================
# ============================================================

ABM_up_DEG <- deg_df %>%
  dplyr::filter(DEG_group == "Higher in ABM") %>%
  dplyr::arrange(celltype_major, p_val_adj, desc(avg_log2FC)) %>%
  dplyr::select(
    celltype_major,
    gene,
    avg_log2FC,
    pct.1,
    pct.2,
    pct_diff,
    p_val,
    p_val_adj,
    DEG_group,
    n_ABM,
    n_Dynamic25d,
    dplyr::everything()
  )

ABM_up_DEG %>%
  dplyr::count(celltype_major, name = "n_ABM_up") %>%
  dplyr::arrange(desc(n_ABM_up))

# ============================================================
# ============================================================

write.csv(
  ABM_up_DEG,
  file = file.path(
    out_dir,
    "Haematopoietic_ABM_up_DEG_all_celltypes.csv"
  ),
  row.names = FALSE
)

saveRDS(
  ABM_up_DEG,
  file = file.path(
    out_dir,
    "Haematopoietic_ABM_up_DEG_all_celltypes.rds"
  )
)


In [ ]:
# ============================================================
# ============================================================

celltypes_use <- unique(ABM_up_DEG$celltype_major)

for (ct in celltypes_use) {

  ct_safe <- gsub("[/ ]", "_", ct)
  ct_safe <- gsub("[^A-Za-z0-9_\\.-]", "_", ct_safe)

  deg_ct <- ABM_up_DEG %>%
    dplyr::filter(celltype_major == ct) %>%
    dplyr::arrange(p_val_adj, desc(avg_log2FC))

  write.csv(
    deg_ct,
    file = file.path(
      out_dir,
      paste0("ABM_up_DEG_", ct_safe, ".csv")
    ),
    row.names = FALSE
  )
}


In [ ]:
# ============================================================
# ============================================================

gene_list_dir <- file.path(out_dir, "gene_lists")
dir.create(gene_list_dir, showWarnings = FALSE, recursive = TRUE)

for (ct in celltypes_use) {

  ct_safe <- gsub("[/ ]", "_", ct)
  ct_safe <- gsub("[^A-Za-z0-9_\\.-]", "_", ct_safe)

  genes_ct <- ABM_up_DEG %>%
    dplyr::filter(celltype_major == ct) %>%
    dplyr::arrange(p_val_adj, desc(avg_log2FC)) %>%
    dplyr::pull(gene) %>%
    unique()

  writeLines(
    genes_ct,
    con = file.path(
      gene_list_dir,
      paste0("ABM_up_genes_", ct_safe, ".txt")
    )
  )
}


In [ ]:
# ============================================================
# ============================================================

ABM_up_HSPC_Ery <- ABM_up_DEG %>%
  dplyr::filter(
    celltype_major %in% c("HSPC/Progenitor", "Erythroid")
  )

write.csv(
  ABM_up_HSPC_Ery,
  file = file.path(
    out_dir,
    "ABM_up_DEG_HSPC_Erythroid.csv"
  ),
  row.names = FALSE
)


In [ ]:
# ============================================================
# Erythroid only: pct_diff - log2FC plot with highlighted genes
# ============================================================

library(dplyr)
library(tidyr)
library(ggplot2)
library(ggrepel)

# ============================================================
# ============================================================

logfc_cutoff <- 1
pct_cutoff <- 0.2

# ============================================================
# ============================================================

deg_df <- Haematopoietic_major_ABM_vs_Dynamic25d_DEG %>%
  dplyr::mutate(
    pct_diff = pct.1 - pct.2,

    DEG_group = dplyr::case_when(
      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC >= logfc_cutoff &
        pct_diff >= pct_cutoff ~ "Higher in ABM",

      !is.na(p_val_adj) & p_val_adj < 0.05 &
        avg_log2FC <= -logfc_cutoff &
        pct_diff <= -pct_cutoff ~ "Higher in Dynamic.25d",

      TRUE ~ "Not significant"
    ),

    DEG_group = factor(
      DEG_group,
      levels = c(
        "Not significant",
        "Higher in ABM",
        "Higher in Dynamic.25d"
      )
    )
  )

table(deg_df$DEG_group)

# ============================================================
# ============================================================

celltype_show <- "Erythroid"

deg_df_one <- deg_df %>%
  dplyr::filter(celltype_major == celltype_show)

table(deg_df_one$DEG_group)

# ============================================================
# ============================================================

deg_count_one <- deg_df_one %>%
  dplyr::count(DEG_group, name = "n_genes") %>%
  tidyr::pivot_wider(
    names_from = DEG_group,
    values_from = n_genes,
    values_fill = 0
  )

if (!"Higher in ABM" %in% colnames(deg_count_one)) {
  deg_count_one$`Higher in ABM` <- 0
}

if (!"Higher in Dynamic.25d" %in% colnames(deg_count_one)) {
  deg_count_one$`Higher in Dynamic.25d` <- 0
}

if (!"Not significant" %in% colnames(deg_count_one)) {
  deg_count_one$`Not significant` <- 0
}

deg_count_one <- deg_count_one %>%
  dplyr::mutate(
    label_ABM = paste0("ABM: ", `Higher in ABM`),
    label_Dynamic = paste0("Dynamic: ", `Higher in Dynamic.25d`)
  )

deg_count_one

# ============================================================
# ============================================================

#genes_to_label <- c(
#  "HBB",
#  "HBD",
#  "LYAR",
#  "MYB"
#)

genes_to_label <- c("HBB", "HBD", "LYAR", "MYB", "STAT5A", "KDM1A", "MEF2C")

label_df <- deg_df_one %>%
  dplyr::filter(gene %in% genes_to_label)

label_df %>%
  dplyr::select(
    gene,
    avg_log2FC,
    pct.1,
    pct.2,
    pct_diff,
    p_val_adj,
    DEG_group
  )

setdiff(genes_to_label, label_df$gene)

# ============================================================
# ============================================================

deg_cols <- c(
  "Higher in ABM"         = "#D95F5F",
  "Higher in Dynamic.25d" = "#6A9BCB"
)

highlight_col <- "#A50026"

# ============================================================
# ============================================================

p_ery <- ggplot() +

  geom_hline(
    yintercept = c(-logfc_cutoff, logfc_cutoff),
    linetype = "dashed",
    linewidth = 0.5,
    color = "grey60"
  ) +

  geom_vline(
    xintercept = 0,
    linetype = "solid",
    linewidth = 0.45,
    color = "grey55"
  ) +

  geom_vline(
    xintercept = c(-pct_cutoff, pct_cutoff),
    linetype = "dashed",
    linewidth = 0.45,
    color = "grey70"
  ) +

  geom_point(
    data = deg_df_one %>% dplyr::filter(DEG_group == "Not significant"),
    aes(x = pct_diff, y = avg_log2FC),
    color = "grey82",
    size = 1.0,
    alpha = 0.55,
    show.legend = FALSE
  ) +

  geom_point(
    data = deg_df_one %>% dplyr::filter(DEG_group == "Higher in Dynamic.25d"),
    aes(x = pct_diff, y = avg_log2FC, color = DEG_group),
    size = 1.4,
    alpha = 0.9
  ) +

  geom_point(
    data = deg_df_one %>% dplyr::filter(DEG_group == "Higher in ABM"),
    aes(x = pct_diff, y = avg_log2FC, color = DEG_group),
    size = 1.4,
    alpha = 0.9
  ) +

  geom_point(
    data = label_df,
    aes(x = pct_diff, y = avg_log2FC),
    color = highlight_col,
    size = 3.0,
    alpha = 1,
    show.legend = FALSE
  ) +

  geom_point(
    data = label_df,
    aes(x = pct_diff, y = avg_log2FC),
    color = highlight_col,
    size = 2.2,
    alpha = 1,
    show.legend = FALSE
  ) +

  geom_text_repel(
    data = label_df,
    aes(
      x = pct_diff,
      y = avg_log2FC,
      label = gene
    ),
    color = highlight_col,
    size = 4.2,
    fontface = "bold",
    box.padding = 0.35,
    point.padding = 0.25,
    segment.color = "grey50",
    segment.size = 0.4,
    max.overlaps = Inf,
    show.legend = FALSE
  ) +

  geom_text(
    data = deg_count_one,
    aes(
      x = 0.95,
      y = Inf,
      label = label_ABM
    ),
    inherit.aes = FALSE,
    hjust = 1,
    vjust = 1.35,
    size = 4.2,
    fontface = "bold",
    color = "#D95F5F"
  ) +

  geom_text(
    data = deg_count_one,
    aes(
      x = -0.95,
      y = -Inf,
      label = label_Dynamic
    ),
    inherit.aes = FALSE,
    hjust = 0,
    vjust = -0.8,
    size = 4.2,
    fontface = "bold",
    color = "#6A9BCB"
  ) +

  scale_color_manual(values = deg_cols) +

  coord_cartesian(
    xlim = c(-1, 1)
  ) +

  labs(
    title = "Erythroid",
    x = expression(Delta~"Percentage Difference"),
    y = "Average log2 Fold Change",
    color = NULL,
    caption = paste0(
      "Colored points indicate genes with FDR-adjusted P value < 0.05, ",
      "|log2FC| >= ", logfc_cutoff,
      ", and |delta  percentage difference| >= ", pct_cutoff, "."
    )
  ) +

  theme_bw(base_size = 15) +
  theme(
    panel.grid = element_blank(),

    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16,
      color = "black"
    ),

    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      color = "black"
    ),
    axis.text.y = element_text(
      color = "black"
    ),
    axis.title = element_text(
      color = "black"
    ),

    legend.position = "top",
    legend.text = element_text(
      size = 11,
      color = "black"
    ),

    panel.border = element_rect(
      color = "black",
      linewidth = 0.6
    ),

    plot.caption = element_text(
      size = 9.5,
      color = "grey30",
      hjust = 0
    ),

    plot.margin = margin(
      t = 5.5,
      r = 12,
      b = 18,
      l = 5.5
    )
  )

p_ery


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "Erythroid_pctdiff_log2fc_labeled_highlight.pdf"),
  plot = p_ery,
  width = 3.5,
  height = 5,
  units = "in",
  dpi = 300,
  device = cairo_pdf
)
